In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/metadata/valid_manifest.jsonl
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/metadata/train_manifest.jsonl
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/metadata/rejected_manifest.jsonl
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/metadata/test_manifest.jsonl
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/metadata/prepared_manifest.jsonl
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/audio/7_seg0009.wav
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/audio/18_seg0001.wav
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/audio/1_seg0003.wav
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/audio/13_seg0000.wav
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/audio/18_seg0000.wav
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/audio/5_seg0009.wav
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/audio/14_seg0000.wav
/kaggle/input/datasets/nguyendangvianh/my-qwen-tts/audio/5_seg0005.wav
/kaggle/input/data

In [8]:
%%bash
# 1. Cài đặt thư viện lõi Qwen3-TTS chính hãng (Bỏ qua flash-attn cài quá lâu)
pip install -U qwen-tts
# 3. Tải bộ công cụ Fine-Tuning của Qwen3
git clone https://github.com/QwenLM/Qwen3-TTS.git
# (Kéo thêm ModelScope dự phòng cho mạng)
pip install -U modelscope

fatal: destination path 'Qwen3-TTS' already exists and is not an empty directory.


In [9]:
import json
import os
import glob
import shutil

# Lấy đường dẫn dataset bạn upload bên cột Input
manifest_paths = glob.glob("/kaggle/input/**/train_manifest.jsonl", recursive=True)
if not manifest_paths:
    print("LỖI KHÔNG TÌM THẤY DỮ LIỆU INPUT! Bạn đã thêm Dataset vào góc phải Notebook chưa?")
else:
    MANIFEST_PATH = manifest_paths[0]
    INPUT_DATA_DIR = os.path.dirname(os.path.dirname(MANIFEST_PATH))
    WORK_DIR = "/kaggle/working/qwen3_finetune"
    os.makedirs(WORK_DIR, exist_ok=True)
    
    # Qwen3-TTS yêu cầu một file ref_audio đại diện chung (Khuyến khích dùng chung 1 giọng duy nhất)
    train_records = []
    first_audio = None
    
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            audio_path = os.path.join(INPUT_DATA_DIR, "audio", os.path.basename(item["audio_filepath"]))
            
            if not first_audio: 
                first_audio = audio_path
                shutil.copy2(first_audio, f"{WORK_DIR}/ref_audio.wav")
                first_audio = f"{WORK_DIR}/ref_audio.wav"
                
            train_records.append({
                "audio": audio_path,
                "text": item["text"],
                "ref_audio": first_audio # Ép dùng chung file chuẩn
            })
            
    # Lưu ra định dạng Qwen chuẩn
    raw_jsonl_path = f"{WORK_DIR}/train_raw.jsonl"
    with open(raw_jsonl_path, "w", encoding="utf-8") as f:
        for r in train_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            
    print("✅ Đã tạo xong train_raw.jsonl chuẩn chỉnh 100% của Qwen3-TTS!")

✅ Đã tạo xong train_raw.jsonl chuẩn chỉnh 100% của Qwen3-TTS!


In [10]:
%%bash
cd /kaggle/working/Qwen3-TTS/finetuning
# Tích hợp token âm thanh bằng Tokenizer chính hãng
python prepare_data.py \
  --device cuda:0 \
  --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz \
  --input_jsonl /kaggle/working/qwen3_finetune/train_raw.jsonl \
  --output_jsonl /kaggle/working/qwen3_finetune/train_with_codes.jsonl


********
********
 


2026-04-18 15:23:04.931217: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776525784.954188     522 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776525784.962041     522 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776525784.981883     522 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776525784.981931     522 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776525784.981937     522 computation_placer.cc:177] computation placer alr

In [16]:
# Bắt máy chủ tải luôn thư mục mô hình về thẳng ổ cứng Kaggle
modelscope download --model Qwen/Qwen3-TTS-12Hz-1.7B-Base...


SyntaxError: invalid decimal literal (4256762115.py, line 2)

In [21]:
%%bash
cd /kaggle/working/Qwen3-TTS/finetuning

# HACK 0: Dọn rác giải phóng phân vùng (Gỡ mìn cái ổ đĩa đang đầy 19.5/19.5GB của bạn)
rm -rf ~/.cache/modelscope/*
rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3-TTS-12Hz-0.6B-Base
rm -rf /kaggle/working/Qwen3-TTS-12Hz-1.7B-Base
rm -rf /kaggle/working/my_qwen_trained_model

# HACK 1: Tắt yêu cầu Flash Attention để tránh bị lỗi cài đặt 30 phút, ép dùng SDPA gốc cực lẹ
sed -i 's/attn_implementation="flash_attention_2",//g' sft_12hz.py

# HACK 2: Fix lỗi phiên bản thư viện Accelerate đòi thư mục Tensorboard trên Kaggle gây crash
sed -i 's/, log_with="tensorboard"//g' sft_12hz.py

# HACK 3: Ép Kaggle tự động gập/căng tần số âm thanh của bạn về 24kHz (Quy định bắt buộc của Qwen3)
sed -i 's/sr=None/sr=24000/g' dataset.py

# HACK 4: Kích hoạt thuật toán nén Tối Ưu Hóa 8-bit (Tiết kiệm 10GB VRAM để T4 không bị cháy)
pip install -U bitsandbytes
sed -i 's/from torch.optim import AdamW/import bitsandbytes as bnb/g' sft_12hz.py
sed -i 's/AdamW(/bnb.optim.Adam8bit(/g' sft_12hz.py

# HACK 5: Lấy đường dẫn vật lý thật của Mô hình bằng Python để không bị trùng lặp file làm bục ổ cứng 19.5 GB
export MODEL_PATH=$(python -c "from huggingface_hub import snapshot_download; print(snapshot_download('Qwen/Qwen3-TTS-12Hz-1.7B-Base'))")

# HACK 6: Chỉ lưu giữ 1 Epoch cuối cùng, xóa sạch Epoch cũ để không bị nhân bản trọng số lên 35.0 GB
sed -i 's@output_dir = os.path.join(args.output_model_path@os.system(f"rm -rf {args.output_model_path}/*"); output_dir = os.path.join(args.output_model_path@g' sft_12hz.py

# Bùng nổ: Training mô hình gốc 1.7B-Base (Hiệu năng tốt nhất, tương thích 100% với file sft_12hz.py)
python sft_12hz.py \
  --init_model_path "$MODEL_PATH" \
  --output_model_path /kaggle/working/my_qwen_trained_model \
  --train_jsonl /kaggle/working/qwen3_finetune/train_with_codes.jsonl \
  --batch_size 1 \
  --lr 2e-6 \
  --num_epochs 10 \
  --speaker_name my_custom_voice


********
********
 
Epoch 0 | Step 0 | Loss: 14.6327
Epoch 0 | Step 10 | Loss: 12.5581
Epoch 0 | Step 20 | Loss: 12.5036
Epoch 0 | Step 30 | Loss: 12.4791
Epoch 0 | Step 40 | Loss: 12.7394
Epoch 0 | Step 50 | Loss: 12.8843
Epoch 0 | Step 60 | Loss: 11.3641
Epoch 1 | Step 0 | Loss: 10.9706
Epoch 1 | Step 10 | Loss: 11.5892
Epoch 1 | Step 20 | Loss: 10.0958
Epoch 1 | Step 30 | Loss: 10.9121
Epoch 1 | Step 40 | Loss: 10.0159
Epoch 1 | Step 50 | Loss: 10.6106
Epoch 1 | Step 60 | Loss: 10.2057
Epoch 2 | Step 0 | Loss: 9.4781
Epoch 2 | Step 10 | Loss: 9.4308
Epoch 2 | Step 20 | Loss: 10.0041
Epoch 2 | Step 30 | Loss: 9.3125
Epoch 2 | Step 40 | Loss: 9.7761
Epoch 2 | Step 50 | Loss: 10.2951
Epoch 2 | Step 60 | Loss: 9.5800
Epoch 3 | Step 0 | Loss: 9.4512
Epoch 3 | Step 10 | Loss: 9.0413
Epoch 3 | Step 20 | Loss: 9.4554
Epoch 3 | Step 30 | Loss: 9.4589
Epoch 3 | Step 40 | Loss: 9.5955
Epoch 3 | Step 50 | Loss: 9.1821
Epoch 3 | Step 60 | Loss: 8.6521
Epoch 4 | Step 0 | Loss: 8.9338
Epoch 4 | S

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 169335.25it/s]
2026-04-18 16:21:25.631697: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776529285.661065    2056 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776529285.670409    2056 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776529285.711387    2056 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776529285.711418    2056 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776

In [23]:
%%bash
cd /kaggle/working
# Dọn dẹp cục zip cũ đang bị kẹt
rm -f my_qwen_trained_model.zip
# Đóng gói với tốc độ tên lửa (tham số -0 tắt ép nén)
zip -r -0 my_qwen_trained_model.zip my_qwen_trained_model/checkpoint-epoch-9


  adding: my_qwen_trained_model/checkpoint-epoch-9/ (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/config.json (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/tokenizer_config.json (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/speech_tokenizer/ (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/speech_tokenizer/config.json (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/speech_tokenizer/configuration.json (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/speech_tokenizer/preprocessor_config.json (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/speech_tokenizer/model.safetensors (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/README.md (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/.gitattributes (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/preprocessor_config.json (stored 0%)
  adding: my_qwen_trained_model/checkpoint-epoch-9/model.safet

In [24]:
from IPython.display import FileLink
FileLink(r'my_qwen_trained_model.zip')


/kaggle/working/my_qwen_trained_model.zip

In [29]:
import sys
import torch
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

# 1. Nạp thẳng Mô hình 3.5GB thần thánh bạn vừa Train xong
model_dir = "/kaggle/working/my_qwen_trained_model/checkpoint-epoch-9"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = Qwen3TTSModel.from_pretrained(
    model_dir, 
    device_map=device, 
    dtype=torch.float32 if device == "cpu" else torch.bfloat16
)

# 2. Xé nháp câu Tiếng Việt để test độ mượt
test_text = "Chào mọi người, đối với tôi, việc học không bao giờ dừng lại ở trường lớp. Mỗi ngày, chỉ cần dành ra 15-30 phút để đọc sách, học một từ mới hay tìm hiểu về một lĩnh vực khác biệt, chúng ta đang tự nâng cấp bản thân. Kiến thức giống như một cây cổ thụ, ngày hôm nay bạn gieo một hạt giống, ngày mai nó sẽ tỏa bóng mát cho tương lai. Đừng sợ chậm, chỉ sợ dừng lại. Hãy học để hiểu, học để làm, và học để sống tốt hơn. Hãy bắt đầu ngay hôm nay nhé!"
print("Đang tổng hợp giọng nói...")

# 3. Yêu cầu AI đọc ra âm thanh
wavs, sr = model.generate_custom_voice(text=test_text, speaker="my_custom_voice")

# 4. Hiển thị cái Vệt Sóng Lõi (Nút Play/Pause Audio) thả trực tiếp lên màn hình Kaggle
display(Audio(wavs[0], rate=sr))


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Đang tổng hợp giọng nói...
